<img src="https://raw.githubusercontent.com/swargo98/adaptiSeq/main/adaptiseq/assets/logo.png" alt="adaptiSeq" width="420">

# adaptiSeq — Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/swargo98/adaptiSeq/blob/main/adaptiSeq_Colab_Quickstart.ipynb)
[![PyPI](https://img.shields.io/pypi/v/adaptiseq.svg)](https://pypi.org/project/adaptiseq/)

Install **[adaptiSeq](https://pypi.org/project/adaptiseq/)** and test its basics —
metadata, single & **batch downloads**, and the Python API — on a small public
dataset that downloads in **seconds**.

- 📖 Docs: https://swargo98.github.io/adaptiSeq/
- 💻 Source: https://github.com/swargo98/adaptiSeq

**To run everything:** *Runtime → Run all*. No GPU needed. The notebook installs adaptiSeq **0.1.12**.

## 1. Install

adaptiSeq is a Python package plus a few external command-line tools it shells out
to. In Colab, `wget` is already present — we just add `pigz` (used by
the CLI `-g` path). `sra-tools` and Aspera are **not** needed for this quickstart.

(Outside Colab, `conda create -n adaptiseq -c conda-forge -c bioconda adaptiseq` installs
adaptiSeq and every external tool from Bioconda in one step.)

In [ ]:
!pip install -q adaptiseq==0.1.15
!apt-get -qq install -y pigz > /dev/null
print("\n✅ install complete")

## 2. Verify the installation

In [ ]:
!adaptiseq --version

import adaptiseq
print("import OK — adaptiseq", adaptiseq.__version__)
print("public API:", [n for n in ("fetch", "resolve", "get_metadata") if hasattr(adaptiseq, n)])

## 3. Fetch metadata (no download)

`-m` resolves an accession across **GSA / SRA / ENA / DDBJ / GEO** and writes a
metadata table — no sequence data is downloaded. We use **PRJNA916347**, a small
project whose runs are only a few KB each.

In [ ]:
!adaptiseq -i PRJNA916347 -m -o demo -Q

import pandas as pd
meta = pd.read_csv("demo/PRJNA916347.metadata.tsv", sep="\t")
print(f"{len(meta)} runs in PRJNA916347")
meta[["run_accession", "fastq_bytes", "fastq_ftp"]].head()

## 4. Batch download (CLI)

The core feature: resolve many accessions **in parallel** and download them through
an **adaptive worker pool**. We take a few of the smallest runs and fetch them as
gzip FASTQ.

Flags: `-g` gzip FASTQ · `-j 4` up to 4 workers · `-k` skip the md5 check · `-Q` quiet.

By default adaptiSeq checks every file's md5 as it lands. The command-line tool then
asks for `sra-tools` at start-up, which Colab does not have, so this cell passes `-k`.
The next section does the verified download from Python, which needs no `sra-tools`.

In [ ]:
runs = meta["run_accession"].head(3).tolist()
with open("batch.txt", "w") as f:
    f.write("\n".join(runs) + "\n")
print("batch list:", runs)

!adaptiseq -i batch.txt -g -k -j 4 -o demo_batch -Q

import os
print("\ndownloaded:", sorted(f for f in os.listdir("demo_batch") if f.endswith(".gz")))

## 5. Batch download (Python API — with md5 verification)

The importable API returns a `FetchResult` and, unlike the `-k` CLI shortcut above,
**md5-verifies** every file against the archive's checksum as it lands, inside the
worker pool (a corrupt copy is deleted and fetched again) — still no `sra-tools`
needed. A run that cannot be delivered and verified is listed in `result.fail_ids`
(and in `fail.log`), never silently skipped. It runs correctly inside Colab's live
event loop.

In [ ]:
import adaptiseq

result = adaptiseq.fetch("batch.txt", outdir="demo_api", gzip=True, jobs=4, adaptive=True, quiet=True)
print("✅ downloaded & md5-verified:", result.success_ids)
print("   failed:", result.fail_ids, "| any failure?", result.failed)

## 6. Single-accession helpers: `resolve` and `get_metadata`

In [ ]:
run = runs[0]

# Where would this run download from, as gzip FASTQ? (no download performed)
print("resolved URL:", adaptiseq.resolve(run, database="ena", gzip=True))

# Parsed metadata rows for a single run:
rows = adaptiseq.get_metadata(run)
print("columns:", len(rows[0]), "| library_strategy:", rows[0].get("library_strategy"))

## 7. Inspect the results

In [ ]:
!ls -la demo_api
print("\n--- success.log ---")
!cat demo_api/success.log
print("\n--- first FASTQ reads ---")
!zcat demo_api/*.fastq.gz 2>/dev/null | head -4

## 8. Speed: adaptiSeq vs Kingfisher on two ~2 GB files

Here we time both tools on the same run, **SRR15852393** (ENA): two gzip FASTQ files of
1.85 GB and 1.89 GB, 3.7 GB in all. Both tools fetch from ENA and check every file's md5.
[Kingfisher](https://github.com/wwood/kingfisher-download) runs with its ENA FTP method
(`-m ena-ftp`, which uses `aria2c`); adaptiSeq runs from Python with its defaults, so it
splits each file into byte ranges and keeps several connections busy per file.

Each tool takes a few minutes and about 4 GB of disk (the files are deleted at the end).
The numbers depend on Colab's network and on how busy ENA is, so they change from run to
run: treat one run as a demonstration, and run the cells again to see the spread.

In [ ]:
!pip install -q kingfisher==0.5.0
!apt-get -qq install -y aria2 > /dev/null
!kingfisher --version

In [ ]:
import glob, hashlib, os, shutil, subprocess, time

ACC = "SRR15852393"
# md5 of the two files, as published by ENA
EXPECTED = {"d300f6eca2d04fad249f50d7e9309a83", "72743762682addb0ada09a77d16acd4e"}
timings = {}

def report(name, outdir, seconds):
    files = sorted(glob.glob(f"{outdir}/**/*.fastq.gz", recursive=True))
    size = sum(os.path.getsize(f) for f in files)
    md5s = set()
    for path in files:                      # checked after the clock has stopped
        h = hashlib.md5()
        with open(path, "rb") as fh:
            for block in iter(lambda: fh.read(1 << 24), b""):
                h.update(block)
        md5s.add(h.hexdigest())
    timings[name] = seconds
    print(f"\n{name}: {len(files)} files, {size/1e9:.2f} GB in {seconds:.0f} s"
          f" = {size/1e6/seconds:.1f} MB/s | md5 matches ENA: {md5s == EXPECTED}")

In [ ]:
# Kingfisher (one run, ENA FTP, md5 check on)
shutil.rmtree("speed_kingfisher", ignore_errors=True)
t0 = time.perf_counter()
try:
    subprocess.run(["kingfisher", "get", "-r", ACC, "-m", "ena-ftp", "--check-md5sums",
                    "-f", "fastq.gz", "--output-directory", "speed_kingfisher"], check=True)
    report("Kingfisher", "speed_kingfisher", time.perf_counter() - t0)
except subprocess.CalledProcessError:
    # ENA's FTP server sometimes stalls a transfer; keep going so adaptiSeq's cell still runs
    print(f"\nKingfisher did not deliver the run after {time.perf_counter() - t0:.0f} s "
          "(its FTP transfer failed), so there is no Kingfisher timing this time.")

In [ ]:
# adaptiSeq (same run, defaults; md5 is checked inside the worker pool)
import adaptiseq

shutil.rmtree("speed_adaptiseq", ignore_errors=True)
t0 = time.perf_counter()
result = adaptiseq.fetch(ACC, outdir="speed_adaptiseq", gzip=True, quiet=True)
report("adaptiSeq", "speed_adaptiseq", time.perf_counter() - t0)
print("failed runs:", result.fail_ids)

In [ ]:
if "Kingfisher" in timings:
    k, a = timings["Kingfisher"], timings["adaptiSeq"]
    print(f"Kingfisher {k:.0f} s   adaptiSeq {a:.0f} s   ->   adaptiSeq took {a/k:.2f}x the time"
          f" ({'faster' if a < k else 'slower'} here)")
else:
    print(f"Kingfisher gave no valid result this time; adaptiSeq took {timings['adaptiSeq']:.0f} s.")

!rm -rf speed_kingfisher speed_adaptiseq   # free the disk

**What we saw.** We ran these same steps on a cloud VM (not Colab) on 2026-11-09, on the same run.
adaptiSeq took 342 s and, in a second run, 148 s; both times the md5 sums matched ENA's.
Kingfisher took 492 s. In another run it took 551 s, but its own check rejected the second
file, so it delivered nothing usable. For the one valid Kingfisher run that is 1.4 to 3.3 times
as long as adaptiSeq. The spread between our two adaptiSeq runs shows how much the speed of
ENA's servers changes from hour to hour, so your numbers will differ; run the cells twice.

In our benchmark (ENA lists, two sites, five repetitions each) the gap was largest on lists of
many small runs and smaller on large files, where Kingfisher also opens several connections:
on lists of large runs the fastest other tool that delivered everything was a median 2.8 to 3.3
times slower than adaptiSeq's default. See the
[comparison page](https://swargo98.github.io/adaptiSeq/comparison/) and the
[adaptive controller notes](https://swargo98.github.io/adaptiSeq/ADAPTIVE_CONTROLLER/).

## Notes & next steps

- **Swap in your own accessions** — any Project / Study / BioSample / Sample /
  Experiment / Run across **GSA / SRA / ENA / DDBJ / GEO**, single or a file of many
  (mixed databases allowed).
- **External tools:** `wget` ships with Colab; we installed `pigz`. For the CLI's md5
  check (without `-k`) and for SRA→FASTQ conversion (`-q`/`-e`), also install
  `sra-tools`: `!apt-get install -y sra-toolkit` (heavier). **Aspera (`-a`) is not
  available in Colab.**
- **Exit status:** on the command line, `0` means every requested run was delivered
  and verified; `1` means at least one was not (it is named in `fail.log`). In
  Python, check `result.failed`.
- **SRA fallback:** a run that ENA cannot serve is fetched from NCBI SRA by default
  (announced; needs `sra-tools`; SRA copies cannot be checked against ENA's md5).
  `--no-sra-fallback` on the command line refuses it.
- **Adaptive controller:** adaptive concurrency is on by default and uses the
  `topdown` controller. Pick another with `--controller topdown|bottomup|greedy|budget|knee`
  (`controller=` in `fetch`), or turn it off with `--no-adaptive -j 20`: on ENA a
  fixed pool is often faster, and the adaptive default needs no value chosen for
  the list. See the docs for how each controller works.
- 📖 Docs: https://swargo98.github.io/adaptiSeq/ · 🐛 Issues:
  https://github.com/swargo98/adaptiSeq/issues